# 集群上运行Manta的流程

## 一、创建环境

### 源码压缩包路径：/mnt/home/ygjx/chenkejin/manta/manta-1.6.0.centos6_x86_64(1).tar.bz2

### 解压缩：tar -xjvf "manta-1.6.0.centos6_x86_64(1).tar.bz2"

### 集群可能无法联网，遂用清华源创建环境：
### conda create -n manta_env -c https://mirrors.tuna.tsinghua.edu.cn/anaconda/cloud/conda-forge/ -c https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main/ -c https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/free/ python=2.7 --override-channels -y

### 进入环境：conda activate manta_env

## 二、运行

### 1、单一配对样本可调用/mnt/home/ygjx/chenkejin/manta/manta-1.6.0.centos6_x86_64/test_single_manta.sh

### test_single_manta.sh脚本如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=manta_1866277           # 作业名称
#SBATCH --nodes=1                          # 申请 1 个节点
#SBATCH --cpus-per-task=24                 # 分配 24 个 CPU
#SBATCH --mem=32G                          # Manta 对内存非常友好，32G 完全足够
#SBATCH --output=/mnt/home/ygjx/chenkejin/Manta/logs/manta_1866277_%j.log

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
# ⚠️ 请确保这个是你新建立的、带有 Python 2.7 的 Manta 环境名称
conda activate manta_env  

# --- 1. 全局变量与路径 (⚠️ 【请修改】这里的内容) ---
# 填写你刚刚解压的 Manta 所在的绝对路径
MANTA_DIR="/mnt/home/ygjx/chenkejin/manta/manta-1.6.0.centos6_x86_64/" 
# Manta 强制要求参考基因组(必需与其配套的 .fai 索引)。如果用的是 hg38，请填上准确路径
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta" 

# --- 2. 样本与数据路径配置 ---
SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/Manta"

PREFIX="1866277"
NORMAL_ID="${PREFIX}N"
TUMOR_ID="${PREFIX}T"
THREADS=24

# 精准获取源文件路径
NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 开始处理 Manta 测试样本: ${PREFIX}"
echo "Tumor BAM:  $TUMOR_BAM"
echo "Normal BAM: $NORMAL_BAM"
echo "=========================================================="

# --- 3. 建立专属沙盒目录 ---
mkdir -p "${WORK_DIR}/logs"
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

# Manta 会把所有中间文件和日志放在这一个文件夹里
# 注意：Manta 要求这个运行目录在开始前不能存在，如果存在它会报错中止，所以我们先强制删除
RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_MantaWorkflow"
rm -rf "$RUN_DIR"  

# --- 4. 构建标准染色体过滤 (借鉴师姐的区域拦截机制) ---
# 避免 Manta 去计算 chrUn, decoy 导致耗时翻倍
STD_CHRS="chr1 chr2 chr3 chr4 chr5 chr6 chr7 chr8 chr9 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 chrX chrY"

REGION_ARGS=""
for chr in $STD_CHRS; do
    REGION_ARGS="$REGION_ARGS --region $chr"
done

# --- 5. 核心流程 Step 1: 配置 Manta 工作流 ---
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/2: 运行 configManta.py 生成配置..."
python "${MANTA_DIR}/bin/configManta.py" \
    --tumorBam "$TUMOR_BAM" \
    --normalBam "$NORMAL_BAM" \
    --referenceFasta "$REF_FA" \
    --runDir "$RUN_DIR" \
    $REGION_ARGS

# --- 6. 核心流程 Step 2: 运行 Manta 推断 ---
# 师姐的脚本里这里加了 "&" 放后台，但在 Slurm 任务里我们不能放后台，否则节点会瞬间判任务结束。
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/2: 运行 runWorkflow.py 开始推断分析..."
python "${RUN_DIR}/runWorkflow.py" -m local -j $THREADS

# --- 7. 提取终极产物 ---
# Manta 的输出极其规范，会自动生成 somatic (肿瘤特有), diploid (生殖系) 两类结果并自动用 bgzip 压缩和 tabix 建库。
# 我们直接把 Somatic SV 拷贝到我们的最终结果夹。
if [ -f "${RUN_DIR}/results/variants/somaticSV.vcf.gz" ]; then
    cp "${RUN_DIR}/results/variants/somaticSV.vcf.gz" "${FINAL_VCF_DIR}/${PREFIX}.manta.somatic.vcf.gz"
    cp "${RUN_DIR}/results/variants/somaticSV.vcf.gz.tbi" "${FINAL_VCF_DIR}/${PREFIX}.manta.somatic.vcf.gz.tbi"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 任务成功！结果已提取至: ${FINAL_VCF_DIR}/${PREFIX}.manta.somatic.vcf.gz"
else
    echo "⚠️ 警告：未找到输出文件 somaticSV.vcf.gz，Manta 运行可能失败。"
    exit 1
fi

### sbatch test_single_manta.sh  运行，tail -f /mnt/home/ygjx/chenkejin/Manta/logs/manta_1866277_1408.log可实时查看运行日志

### 2、批量运行脚本可调用：/mnt/home/ygjx/chenkejin/manta/manta-1.6.0.centos6_x86_64/batch_manta.sh

### batch_manta.sh代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=Manta_Batch             # 作业名称
#SBATCH --nodes=1                          # 每个子任务申请 1 个节点
#SBATCH --cpus-per-task=16                  # 每个子任务分配 16 个 CPU (15个并发共240核)
#SBATCH --mem=32G                          # 每个子任务分配 32G 内存
#SBATCH --array=1-80%15                    # 【核心并发控制】共 80 对样本，每次最多同时运行 15 个
#SBATCH --output=/mnt/home/ygjx/chenkejin/Manta/logs/slurm_array_%A_%a.out 

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate manta_env

# --- 1. 全局变量与路径 ---
MANTA_DIR="/mnt/home/ygjx/chenkejin/manta/manta-1.6.0.centos6_x86_64/"
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
THREADS=8

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/Manta"
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"

# 总体进度汇报日志
MASTER_LOG="${WORK_DIR}/master_progress_manta.log"

# --- 2. 任务解析 ---
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST")
NORMAL_ID=$(echo "$LINE" | awk '{print $1}')
TUMOR_ID=$(echo "$LINE" | awk '{print $2}')

PREFIX=$(echo "$NORMAL_ID" | sed 's/N//')

if [ -z "$PREFIX" ]; then
    exit 0
fi

# --- 3. 专属日志动态重定向 ---
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.log"
exec > >(tee -i "$SAMPLE_LOG") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] 样本 ${PREFIX} 开始处理"
echo "任务阵列 ID: ${SLURM_ARRAY_TASK_ID}/80  执行节点: $(hostname)"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX}" >> "$MASTER_LOG"

# --- 4. 建立专属小沙盒与结果目录 ---
RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_MantaWorkflow"
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

# 定义最终输出文件路径
vcf_gz_output="${FINAL_VCF_DIR}/${PREFIX}.manta.somatic.vcf.gz"
vcf_unzipped_output="${FINAL_VCF_DIR}/${PREFIX}.manta.somatic.vcf"

# 【断点续传修改】现在检查的是解压后的纯文本 .vcf 是否存在
if [ -f "$vcf_unzipped_output" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] 样本 ${PREFIX} 已存在解压结果，跳过运行。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Sample ${PREFIX} skipped" >> "$MASTER_LOG"
    exit 0
fi

rm -rf "$RUN_DIR"

# --- 5. 精准拉取数据文件 ---
NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# --- 6. 核心流程执行 ---
{
    STD_CHRS="chr1 chr2 chr3 chr4 chr5 chr6 chr7 chr8 chr9 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 chrX chrY"
    REGION_ARGS=""
    for chr in $STD_CHRS; do
        REGION_ARGS="$REGION_ARGS --region $chr"
    done

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 配置 Manta 工作流..."
    python "${MANTA_DIR}/bin/configManta.py" \
        --tumorBam "$TUMOR_BAM" \
        --normalBam "$NORMAL_BAM" \
        --referenceFasta "$REF_FA" \
        --runDir "$RUN_DIR" \
        $REGION_ARGS

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 Manta 核心计算引擎..."
    python "${RUN_DIR}/runWorkflow.py" -m local -j $THREADS

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 提取并解压终极产物..."
    if [ -f "${RUN_DIR}/results/variants/somaticSV.vcf.gz" ]; then
        # 1. 拷贝原始的压缩包和索引
        cp "${RUN_DIR}/results/variants/somaticSV.vcf.gz" "$vcf_gz_output"
        cp "${RUN_DIR}/results/variants/somaticSV.vcf.gz.tbi" "${vcf_gz_output}.tbi"
        
        # 2. 【新增】立即解压出纯文本的 VCF 文件
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] 正在执行解压操作..."
        gunzip -c "$vcf_gz_output" > "$vcf_unzipped_output"
        
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 运行且解压成功！"
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} completed" >> "$MASTER_LOG"
        
        # 清理沙盒
        rm -rf "$RUN_DIR"
    else
        echo "⚠️ 错误：未找到预期的输出文件 somaticSV.vcf.gz"
        false 
    fi

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 样本 ${PREFIX} 运行失败！"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} failed!" >> "$MASTER_LOG"
    exit 1
}

### sbatch batch_manta.sh运行，tail -f /mnt/home/ygjx/chenkejin/Manta/master_progress_manta.log可实时查看运行日志

In [ ]:
结果路径：/mnt/home/ygjx/chenkejin/Manta/Final_Results/